In [ ]:
!git clone https://github.com/baasmaala/road-safety-explorer.git
%cd road-safety-explorer

# Notebook 01 — Cleaning the WHO data and finding country clusters

This is the first notebook in the pipeline. It takes the raw WHO Global Status Report on Road Safety 2023 spreadsheet and turns it into something the rest of the project can actually use.

Two things happen here:

1. **Cleaning.** The raw file has two header rows, inconsistent country names, and a lot of columns we don't need. We sort that out.
2. **Clustering.** Once the country-level features are clean, we run K-Means to group countries into road-safety profiles. The output of this notebook feeds directly into my teammates Omar's PCA notebook (`02_pca_anomaly.ipynb`).

Palestine appears in the WHO file as *"occupied Palestinian territory, including east Jerusalem"*. We rename it to **Palestine** so the rest of the pipeline (and the time-series notebook, which already uses "Palestine") line up.

**Outputs of this notebook** — both saved to `data/processed/`:
- `country_features.csv` — the clean numeric feature matrix per country
- `country_clusters.csv` — country, cluster label, and the headline metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


REPO = Path("road-safety-explorer") if Path("road-safety-explorer").exists() else Path(".")
RAW = REPO / "data" / "raw"
PROCESSED = REPO / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

WHO_FILE = RAW / "gsrrs23-indicators-for-participating-countries-or-territories.xlsx"


sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("Pandas :", pd.__version__)
print("Reading from:", WHO_FILE)
print("Writing to  :", PROCESSED)

## 1. Look at the raw file before touching it

Before any cleaning, just open the file and see what we're dealing with. The WHO release is a single spreadsheet with two sheets: `Indicators` (the actual data) and `Notes` (metadata about the columns).

The thing to notice — and the reason we can't just call `pd.read_excel(...)` and walk away — is that the `Indicators` sheet has **two header rows**:

- Row 0 = short variable codes (`CP_3`, `CP_19`, `CP_23` …)
- Row 1 = human-readable descriptions (`Population`, `WHO-estimated road traffic fatalities`, …)
- Row 2 onwards = the actual country data

If we read it normally pandas will treat row 0 as the header and shove row 1 (the descriptions) into the data, which will turn every numeric column into text and quietly break everything. So step one is to read the file *raw* and see the mess for ourselves.

In [ ]:
raw = pd.read_excel(WHO_FILE, sheet_name="Indicators", header=None)

print("Raw shape :", raw.shape)
print("Sheets    :", pd.ExcelFile(WHO_FILE).sheet_names)
print()
print("First 3 rows × first 6 columns:")
raw.iloc[:3, :6]

## 2. Fix the two-row header

We use the variable codes (row 0) as the column names because they're short, unique, and stable. The descriptions (row 1) are useful for reading the data but they'd be terrible column names — long, full of spaces and special characters.

But we don't throw the descriptions away. We stash them in a separate dictionary (`code_to_label`) so later, when we plot or print results, we can show the friendly name (`"WHO fatality rate /100k"`) instead of the raw code (`"CP_23"`).

Then we drop both header rows from the data and reset the index, so what's left is a clean 173-row table where each row is one country.

In [ ]:
# Row 0 = codes (will become column names)
# Row 1 = human descriptions (kept aside as a lookup dictionary)
codes = raw.iloc[0].astype(str).tolist()
labels = raw.iloc[1].tolist()

code_to_label = {
    code: (str(label) if pd.notna(label) else code)
    for code, label in zip(codes, labels)
}

# Drop the two header rows, set proper column names, reset the index.
df = raw.iloc[2:].copy()
df.columns = codes
df = df.reset_index(drop=True)

print("Cleaned shape:", df.shape)
print()
print("First 3 countries, a few key columns:")
df[["CP_0", "CP_1", "CP_3", "CP_5", "CP_7", "CP_23"]].head(3)

## 3. Normalize country names — find Palestine

We need country names to be consistent across the three notebooks (WHO data here, IHME time-series in Omar's notebook, anomaly results in Osama's). The good news: WHO and IHME use roughly standard names for almost every country. The one exception worth handling is Palestine — WHO labels her *"occupied Palestinian territory, including east Jerusalem"* while the IHME file just says *"Palestine"*.

We define `COUNTRY_NAME_MAP` as a small dictionary covering Palestine's known aliases. The `.replace()` call runs against every row, but in practice only Palestine matches anything — every other country was already using the same name in both files. The dictionary stays in the project as a safety net: if Osama or Omar later spot another mismatch (say, "Türkiye" vs "Turkey"), we add it here and every notebook stays in sync.

Note that the actual *join key* between datasets is the ISO 3-letter code (`CP_0`) — `PSE`, `USA`, `DEU`, etc. ISO codes are unambiguous and don't change. The country name is just for display.

In [ ]:
# Single source of truth for country-name reconciliation across the project.
# Add to this dict if we discover other mismatches later.
COUNTRY_NAME_MAP = {
    "occupied Palestinian territory, including east Jerusalem": "Palestine",
    "occupied Palestinian territory": "Palestine",
    "State of Palestine": "Palestine",
    "Palestinian Territory": "Palestine",
    "West Bank and Gaza Strip": "Palestine",
}

df["CP_1"] = df["CP_1"].replace(COUNTRY_NAME_MAP)

# Sanity check — Palestine must be present, and exactly once.
pal_mask = df["CP_1"] == "Palestine"
print(f"Palestine rows found: {pal_mask.sum()}")

assert pal_mask.sum() == 1, "Expected exactly one Palestine row"

# Pull her row and show the headline numbers.
pal = df[pal_mask].iloc[0]
print(f"\nPalestine snapshot:")
print(f"  ISO code            : {pal['CP_0']}")
print(f"  Population          : {int(pal['CP_3']):,}")
print(f"  Income group        : {pal['CP_5']}")
print(f"  WHO region          : {pal['CP_7']}")
print(f"  Fatality rate /100k : {pal['CP_23']}")
print(f"  WHO-est. fatalities : {int(pal['CP_19'])}")

### What we got

One Palestine row, ISO code `PSE`, population just over 5 million, lower middle income, Eastern Mediterranean Region. The fatality rate of **4.7 per 100k is unusually low** — the global average sits around 13–15 and most of Palestine's regional neighbors are in the 15–25 range. That's a result worth coming back to once we have the cluster labels.

## 4. Audit missingness before picking features

The dataset has 228 columns but most of them aren't useful for clustering — some are administrative (which year of the survey a country participated in), some are footnotes, and many are sparse (only a fraction of countries reported them).

Rather than guessing, we audit. We pick a shortlist of indicators that *could* be useful for road safety clustering, then check how many countries actually have data for each one. Anything below ~75% coverage gets dropped, because imputing values for a third of our rows would just inject noise into the clusters.

This is the boring step that decides whether the clustering means anything.

In [ ]:
# Candidate features for clustering — chosen because each one captures a
# distinct dimension of road safety: exposure, outcomes, who dies, infrastructure rules.
candidate_features = {
    "CP_3":   "Population",
    "CP_19":  "WHO-estimated fatalities (count)",
    "CP_23":  "Fatality rate per 100k",
    "CP_18a": "% deaths: light vehicles",
    "CP_18b": "% deaths: 2/3-wheelers",
    "CP_18c": "% deaths: pedestrians",
    "CP_18d": "% deaths: cyclists",
    "CP_18e": "% deaths: other",
    "CP_37":  "Vehicles per 100k",
    "CP_68":  "Max urban speed limit",
    "CP_69":  "Max rural speed limit",
    "CP_70":  "Max motorway speed limit",
    "CP_76":  "BAC limit (general)",
    "CP_95a": "Helmet wear rate (driver)",
    "CP_100": "Seat-belt rate (drivers)",
    "CP_101": "Seat-belt rate (front)",
    "CP_102": "Seat-belt rate (rear)",
}

# For each candidate, count how many countries have a real numeric value.
n_countries = len(df)
audit_rows = []
for code, label in candidate_features.items():
    series = pd.to_numeric(df[code], errors="coerce")
    n_valid = series.notna().sum()
    audit_rows.append({
        "code": code,
        "indicator": label,
        "non_null": n_valid,
        "coverage_%": round(100 * n_valid / n_countries, 1),
    })

audit = pd.DataFrame(audit_rows).sort_values("coverage_%", ascending=False)
print(f"Total countries: {n_countries}")
print()
audit

### What the audit tells us

There's an obvious cut at the 75% line. Everything above it (13 indicators) is reported by at least 126 of 172 countries, which is workable. Everything below it (4 indicators — helmet wear, seat-belt rates) is missing for over half the world, which would force us to impute values for a third or more of our rows. That's not cleaning, that's making things up. So we drop them.

Looking at what survives, we end up with a balanced feature set — and that matters for clustering, because K-Means will weight every dimension we give it:

- **Outcomes** — fatality rate, fatality count, population. *What's the size of the problem?*
- **Who dies** — five percentages (light vehicles, 2-wheelers, pedestrians, cyclists, other). *What kind of road safety problem is it?* A country where 50% of deaths are pedestrians has a fundamentally different problem than one where 80% are drivers.
- **Rules** — three speed limits + the drink-drive BAC limit. *What's the legal environment?*
- **Exposure** — vehicles per 100k people. *How many cars are even on the road?*

That's outcomes, behavior, environment, and exposure all represented — no single dimension dominates. CP_70 (motorway speed limit) is borderline at 73% coverage, but it's a real road-safety lever and dropping it would leave us with no information about high-speed roads at all. Worth the small imputation cost.

We're deliberately leaving out **WHO Region**. It's tempting to include but it would just teach K-Means to rediscover continents, which isn't insight — it's geography we already know. **Income group**, on the other hand, is a real driver of road safety outcomes and isn't fully implied by the numeric features, so we'll add it as an ordinal encoding when we build the matrix.

## 5. Build the feature matrix

Now we assemble the actual table that K-Means will see. Three things happen here:

1. **Pull the 13 numeric features** and rename them from cryptic codes (`CP_23`) to readable names (`fatality_rate_per_100k`). Future-us will thank present-us when we're plotting cluster profiles.
2. **Encode income group** as an ordinal variable (0 = Low income, 3 = High income). It's not numeric in the raw file but it has a natural order, so an ordinal encoding is more honest than one-hot.
3. **Impute the few remaining missing values** with each feature's median. We're not imputing much — every feature we kept has at least 73% coverage — but K-Means can't handle NaNs.

Index by ISO code (`PSE`, `USA`, etc.) — that's our stable join key for the rest of the project.

In [ ]:
# The 13 features that survived the audit, with readable names.
FEATURES = {
    "CP_3":   "population",
    "CP_19":  "fatalities_count",
    "CP_23":  "fatality_rate_per_100k",
    "CP_18a": "pct_deaths_light_vehicles",
    "CP_18b": "pct_deaths_2_3_wheelers",
    "CP_18c": "pct_deaths_pedestrians",
    "CP_18d": "pct_deaths_cyclists",
    "CP_18e": "pct_deaths_other",
    "CP_37":  "vehicles_per_100k",
    "CP_68":  "speed_limit_urban",
    "CP_69":  "speed_limit_rural",
    "CP_70":  "speed_limit_motorway",
    "CP_76":  "bac_limit_general",
}

# Income group has a natural order, so ordinal encoding is the honest choice.
INCOME_ORDER = {
    "Low income": 0,
    "Lower middle income": 1,
    "Upper middle income": 2,
    "High income": 3,
}

# Start with country identifiers we want to keep alongside the features.
features = pd.DataFrame({
    "iso":          df["CP_0"],
    "country":      df["CP_1"],
    "region":       df["CP_7"],
    "income_group": df["CP_5"],
})

# Add the 13 numeric features, coercing anything non-numeric to NaN.
for code, name in FEATURES.items():
    features[name] = pd.to_numeric(df[code], errors="coerce")

# Add the encoded income group.
features["income_ordinal"] = features["income_group"].map(INCOME_ORDER)

# ISO code becomes the index — stable, unique, our join key for the rest of the project.
features = features.set_index("iso")

# How much imputation are we actually doing?
missing_per_col = features[list(FEATURES.values()) + ["income_ordinal"]].isna().sum()
print("Missing values per feature (before imputation):")
print(missing_per_col[missing_per_col > 0].sort_values(ascending=False))
print()

# Median imputation — column-wise, only on the numeric features.
numeric_cols = list(FEATURES.values()) + ["income_ordinal"]
features[numeric_cols] = features[numeric_cols].fillna(features[numeric_cols].median())

print(f"Final shape: {features.shape}")
print(f"Any NaNs left in numeric features? {features[numeric_cols].isna().any().any()}")
print()
print("Palestine row:")
features.loc["PSE"]

### What the imputation list tells us

The columns that needed the most filling-in tell their own story. **Motorway speed limit was missing for 46 countries** — many of them probably don't have motorways at all, which makes "median imputation" a compromise rather than a true fix. Same logic for vehicles per 100k (36 missing) and BAC limit (38 missing) — these aren't random gaps, they're countries where the indicator either doesn't apply or wasn't reported. We accept this trade-off here so K-Means has something to work with, but it's the kind of thing we should mention in the README's limitations section.

For Palestine specifically, her BAC limit was originally NaN and got filled with the global median (0.05). Defensible, but worth flagging. Everything else in her row is real reported data.

The headline pattern in Palestine's row: **48.92% of road deaths are pedestrians**, while motorcycles and cyclists combined are under 2%. That's a distinctive signature — most countries with low fatality rates are wealthy with strong enforcement (Sweden, Norway). Palestine has a low rate but a very pedestrian-heavy death distribution. That's the road-safety profile of a place where people walk a lot and where motorcycle culture is small. The clustering should pick this up.

## 6. Standardize before clustering

K-Means works on Euclidean distance, which means a feature with a big numeric range will dominate the clustering. Look at our features:

- `population` ranges from ~10,000 to over 1.4 billion
- `bac_limit_general` ranges from 0 to 0.08
- `pct_deaths_pedestrians` ranges from 0 to 100

If we feed these in raw, K-Means will essentially cluster countries by population size and ignore everything else. We standardize each feature to mean 0, standard deviation 1, so every dimension gets equal say in the distance calculation.

We also drop `population` and `fatalities_count` from the *clustering* features themselves — they're absolute scale variables that just measure country size, not road-safety profile. We keep them in the dataset for context (they'll show up when we profile the clusters), but we don't cluster on them. The fatality *rate per 100k* already captures the safety dimension in a population-normalized way.

In [ ]:
# Features we actually cluster on — drop absolute-scale variables.
# Population and fatalities_count are kept in `features` for context but excluded here.
CLUSTER_FEATURES = [
    "fatality_rate_per_100k",
    "pct_deaths_light_vehicles",
    "pct_deaths_2_3_wheelers",
    "pct_deaths_pedestrians",
    "pct_deaths_cyclists",
    "pct_deaths_other",
    "vehicles_per_100k",
    "speed_limit_urban",
    "speed_limit_rural",
    "speed_limit_motorway",
    "bac_limit_general",
    "income_ordinal",
]

X = features[CLUSTER_FEATURES].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Wrap back into a DataFrame so we keep the country index and feature names.
X_scaled = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

print(f"Clustering on {X_scaled.shape[1]} features × {X_scaled.shape[0]} countries")
print()
print("After scaling — every feature should have mean ≈ 0, std ≈ 1:")
X_scaled.describe().loc[["mean", "std"]].round(2)

### Sanity check passed

Every feature now has mean ≈ 0 and std ≈ 1 — that's exactly what we wanted. K-Means will now treat each of the 12 features as equally important when measuring distance between countries. Without this step, population and vehicles-per-100k would have crushed everything else by sheer numeric range.

Worth noticing: we cluster on **12 features**, not the 13 we engineered. Population was dropped because clustering on country *size* would just give us "big countries vs small countries" — that's not insight, it's a list. The fatality *rate* already tells us about safety in a size-normalized way, which is what we want.

## 7. Choose k — how many clusters?

K-Means needs us to tell it upfront how many clusters to look for. There's no formula that gives the "right" answer — it's a judgment call we have to defend. To make the call honestly, we use two diagnostics together:

- **The elbow method** — we run K-Means for k = 2, 3, …, 10 and plot the within-cluster sum of squares (inertia). As k grows, inertia always drops, but at some point adding another cluster stops giving us much. That bend in the curve is the "elbow."
- **The silhouette score** — measures how well each country fits its own cluster compared to the next-nearest cluster. It ranges from −1 (badly placed) to +1 (well separated), and higher is better.

The two methods often disagree slightly. When they do, we pick a value that's reasonable on both *and* that produces clusters we can actually interpret — statistical optimality means nothing if the result is "five almost-identical groups" or "one giant cluster and four small ones."

In [ ]:
# Sweep k from 2 to 10 and record both diagnostics.
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Plot both side by side so we can read them together.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_range), inertias, marker="o", color="#2c3e50")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia (within-cluster SSE)")
axes[0].set_title("Elbow method — look for the bend")
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), silhouettes, marker="o", color="#c0392b")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette — higher is better")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print the numbers too — the chart is for intuition, the table is for defending the choice.
diag = pd.DataFrame({
    "k": list(k_range),
    "inertia": [round(x, 1) for x in inertias],
    "silhouette": [round(x, 3) for x in silhouettes],
})
diag

### Reading the diagnostics — and trying it out

Both methods point to a similar place. The elbow shows the steepest gains happen between k=2 and k=4, then the curve flattens. The silhouette has near-tied peaks at k=4 (0.197) and k=7 (0.198), with k=5 slightly lower (0.174).

We initially tried **k=4** because it sat at the elbow and tied for the silhouette peak. But when we looked at the result, Palestine landed in a 67-country mega-cluster spanning fatality rates from 4.7 to 37.4 — that's everyone from Palestine to Guinea bundled together as one "type." Statistically defensible, but not insightful.

So we tried **k=5**. The silhouette score is slightly lower (0.174 vs 0.197) but the structural improvement is real: the broad developing-country cluster split into a moderate-fatality group (where Palestine landed) and a high-fatality group (Guinea, Libya, Haiti). Palestine's cluster is now 49 countries with fatality rates 4.7–17, which is genuinely interpretable — and Palestine being the lowest in her cluster becomes a cleaner finding because she's now compared against actual peers (Lebanon, Egypt, Tunisia) instead of being lumped with countries 5–7x worse.

We pick **k = 5**. The lesson worth remembering: silhouette and elbow tell us about statistical separation, but they can't tell us whether the resulting clusters mean anything. Both have to be checked together.

## 8. Run K-Means with k = 5

Now the actual clustering. We fit K-Means with `k=5`, attach the cluster label to each country, and immediately do two sanity checks:

1. **Cluster sizes** — are the five groups reasonably balanced, or did we get one giant cluster and four tiny ones?
2. **Where does Palestine land?** — and which countries are her cluster-mates?

`random_state=42` so we get the same labels every time we re-run the notebook (K-Means starts from random initial centers, so without a seed the cluster numbers would shuffle on every run — the *groupings* would be the same but cluster "0" might become cluster "2", which would break every chart downstream).

In [ ]:
# Final model — k=5, fixed seed for reproducibility.
final_km = KMeans(n_clusters=5, n_init=10, random_state=42)
features["cluster"] = final_km.fit_predict(X_scaled)

# How big is each cluster?
print("Cluster sizes:")
print(features["cluster"].value_counts().sort_index())
print()

# Where did Palestine land?
pal_cluster = features.loc["PSE", "cluster"]
print(f"Palestine is in cluster {pal_cluster}")
print()

# Who else is in Palestine's cluster?
mates = features[features["cluster"] == pal_cluster][["country", "region", "income_group", "fatality_rate_per_100k"]]
print(f"Palestine's {len(mates)} cluster-mates:")
mates.sort_values("fatality_rate_per_100k")

### Where Palestine landed — and what it tells us

Five clusters, sizes 17, 24, 29, 49, 53 — reasonably balanced, no degenerate "one giant cluster" or "single-country cluster" weirdness. K-Means found real structure.

Palestine sits in **cluster 3** with 48 other countries. Her cluster-mates include Egypt (9.4), Lebanon (9.7), Tunisia (16.3), Mauritania (9.5), Mongolia (12.4), Rwanda (11.6), and Nigeria (17.2). The fatality rates inside the cluster span **4.7 to ~17 per 100k** — and Palestine sits at the very bottom of that range. She has the lowest fatality rate of any country in her cluster.

That's an interesting finding on its own, but the clustering is doing more than just grouping by fatality rate. If it were, Palestine would have ended up with Sweden, Norway, and the UK (who all have rates around 2-5). The fact that K-Means put her with Egypt and Lebanon instead means it picked up on something deeper — the *shape* of her road safety profile (pedestrian-heavy deaths, lower-middle-income context, similar vehicle/infrastructure patterns) matters more than the headline number alone.

We'll see exactly what defines each cluster in the next step, when we profile their average characteristics.

## 9. Profile each cluster — what makes them different?

Five cluster numbers (0, 1, 2, 3, 4) mean nothing by themselves. To make them useful, we need to look at the *average* feature values inside each cluster and figure out what story each one tells.

We compute the mean of every clustering feature within each cluster, plus the average income level and a few headline numbers (population, fatality count). Then we read across the rows and ask: what's distinctive about this group?

This is where the clustering either pays off or doesn't. If we can give each cluster a real-world description — "high-income, low-fatality, motorcycle-light" — we have something to talk about. If the profiles look indistinguishable, we'd need to revisit our feature choices.

In [ ]:
# Average feature values per cluster — this is the interpretation table.
profile_features = [
    "fatality_rate_per_100k",
    "pct_deaths_pedestrians",
    "pct_deaths_2_3_wheelers",
    "pct_deaths_light_vehicles",
    "pct_deaths_cyclists",
    "vehicles_per_100k",
    "speed_limit_urban",
    "speed_limit_motorway",
    "bac_limit_general",
    "income_ordinal",
]

profiles = features.groupby("cluster")[profile_features].mean().round(2)

# Add cluster size as the first column.
profiles.insert(0, "n_countries", features["cluster"].value_counts().sort_index())

# Add the dominant income group per cluster (mode).
profiles["dominant_income"] = (
    features.groupby("cluster")["income_group"]
    .agg(lambda s: s.mode().iloc[0])
)

# Add 3 representative countries per cluster (lowest, median, highest fatality rate).
def representatives(group):
    sorted_g = group.sort_values("fatality_rate_per_100k")
    low = sorted_g.iloc[0]["country"]
    mid = sorted_g.iloc[len(sorted_g) // 2]["country"]
    high = sorted_g.iloc[-1]["country"]
    return f"{low}, {mid}, {high}"

profiles["examples (low → high fatality)"] = (
    features.groupby("cluster").apply(representatives)
)

print("Cluster profiles — average feature values:")
profiles

### Naming the clusters

Each cluster has a clear personality once you read across the row. We can give each one a real-world name based on what defines it:

- **Cluster 0 — "Upper-middle motorized" (29 countries).** Upper-middle income, very high vehicle ownership (36,700/100k), moderate fatality (15), balanced death distribution. Latin American and island nations dominate — Costa Rica, Dominican Republic, Maldives.

- **Cluster 1 — "High-income safe" (53 countries).** This is the wealthy world. Lowest fatality rate by far (6.1), highest vehicle ownership (55,000/100k), highest motorway speed limits (118 km/h). Mostly European countries plus high-income Asia. They've solved most of the road safety problem.

- **Cluster 2 — "Pedestrian-vulnerable Mediterranean/Middle East" (24 countries).** High pedestrian share of deaths (29%), almost no motorcycle deaths (1.9%), upper-middle income. Cyprus, Jordan, Yemen — a Mediterranean / Middle Eastern walking-and-driving profile.

- **Cluster 3 — Palestine's cluster — "Developing pedestrian-heavy" (49 countries).** Even more pedestrian-dominated than cluster 2 (32% of deaths), lower-middle income, higher fatality rate (19). This is the largest cluster and groups countries where walking is a major mobility mode but road infrastructure hasn't caught up. Palestine, Bolivia, Egypt, Tunisia, Lebanon, Mongolia all live here.

- **Cluster 4 — "Mixed motorized" (17 countries).** Smallest cluster, more motorcycle deaths than the others (8%), high vehicle ownership but lower-middle income on average. The most heterogeneous of the five — likely countries where the imputation step pulled them toward the middle. We flag this in the limitations note: a few countries (Australia is one) appear here partly because they had many missing indicators that got median-imputed, which dampened their distinctive features.

**The Palestine story is now clear.** Her cluster is defined by lots of pedestrian deaths, lower-middle income, and developing road infrastructure. Within that group, **she has the lowest fatality rate of any of the 49 countries** (4.7 vs the cluster average of 19) — which means whatever she's doing differently from her structural peers is working. That's the kind of finding the Streamlit app can highlight on Page 2 (Country Comparison).

## 9.5 Visualize the clusters

The profile table tells us what each cluster looks like, but a table is hard to *feel*. Three visualizations make the structure obvious:

1. **A radar chart per cluster** — five overlapping shapes showing each cluster's "fingerprint" across the key features. The eye picks up differences in shape much faster than differences in numbers.
2. **A strip plot of Palestine's cluster** — all 49 countries in cluster 3 plotted by fatality rate, with Palestine marked. This puts our spotlight finding into a single image.
3. **A heatmap of cluster signatures** — every cluster × every feature, with cells colored by how far above/below the global average each value is. This is the master reference chart.

These same three visuals will carry into the Streamlit app, so the work isn't wasted.

In [ ]:
# Radar chart — one shape per cluster, showing each cluster's average profile.
# Features get min-max normalized to 0-1 so they share a scale on the radar.

from math import pi

radar_features = [
    "fatality_rate_per_100k",
    "pct_deaths_pedestrians",
    "pct_deaths_2_3_wheelers",
    "pct_deaths_light_vehicles",
    "vehicles_per_100k",
    "speed_limit_motorway",
    "bac_limit_general",
]

radar_labels = [
    "Fatality rate",
    "% pedestrian\ndeaths",
    "% 2-wheeler\ndeaths",
    "% light vehicle\ndeaths",
    "Vehicles\nper 100k",
    "Motorway\nspeed limit",
    "BAC limit",
]

cluster_means = features.groupby("cluster")[radar_features].mean()
normed = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

n = len(radar_features)
angles = [k / float(n) * 2 * pi for k in range(n)]
angles += angles[:1]

cluster_colors = {
    0: "#3498db",   # blue
    1: "#2ecc71",   # green
    2: "#9b59b6",   # purple
    3: "#e74c3c",   # red — Palestine's cluster
    4: "#f39c12",   # orange
}

CLUSTER_NAMES = {
    0: "Upper-middle motorized",
    1: "High-income safe",
    2: "Pedestrian-vulnerable Med/ME",
    3: "Developing pedestrian-heavy ★",   # ★ = Palestine
    4: "Mixed motorized",
}

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for cluster_id in sorted(normed.index):
    values = normed.loc[cluster_id].tolist()
    values += values[:1]
    color = cluster_colors[cluster_id]
    is_palestine = (cluster_id == 3)

    ax.plot(angles, values, color=color,
            linewidth=2.5 if is_palestine else 1.5,
            label=f"Cluster {cluster_id}: {CLUSTER_NAMES[cluster_id]}")
    ax.fill(angles, values, color=color, alpha=0.18 if is_palestine else 0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=10)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(["", "", ""])
ax.set_ylim(0, 1)

plt.title("Cluster fingerprints — each shape is one road safety profile",
          size=13, pad=25, weight="bold")
plt.legend(loc="upper right", bbox_to_anchor=(1.35, 1.05), fontsize=9)
plt.tight_layout()
plt.show()

### What the radar tells us

Each cluster has a different shape, which is the visual proof that K-Means found real structure.

- **Cluster 1 (green, high-income safe)** is the smallest shape — low on fatality rate, low on the death-share dimensions, balanced overall. This is what "the road safety problem mostly solved" looks like.
- **Cluster 0 (blue, upper-middle motorized)** stretches toward vehicles per 100k and BAC — high motorization, moderate fatality rate.
- **Cluster 2 (purple, pedestrian-vulnerable Med/ME)** spikes on pedestrian deaths and motorway speed limit, dips on 2-wheeler deaths.
- **Cluster 3 (red — Palestine)** is the widest shape on pedestrian deaths and fatality rate. This is the developing-pedestrian-heavy signature.
- **Cluster 4 (orange, mixed motorized)** is the most balanced/middling shape, which is consistent with it being the smallest and most heterogeneous cluster.

Palestine's cluster doesn't just have *more* pedestrian deaths than others — it's the defining feature of the group. Whatever road safety story we tell about Palestine has to start with pedestrians.

In [ ]:
# Strip plot — all countries in Palestine's cluster, ranked by fatality rate.
# Goal: show in one image that Palestine has the lowest rate of any country in her cluster.

palestine_cluster = features.loc["PSE", "cluster"]
mates = features[features["cluster"] == palestine_cluster].copy()
mates = mates.sort_values("fatality_rate_per_100k").reset_index()

fig, ax = plt.subplots(figsize=(11, 4.5))

# Highlight Palestine vs the rest.
is_palestine = mates["country"] == "Palestine"
ax.scatter(mates.loc[~is_palestine, "fatality_rate_per_100k"],
           [0] * (~is_palestine).sum(),
           s=70, color="#95a5a6", alpha=0.55, edgecolor="white", linewidth=0.6,
           label=f"Other countries in cluster (n={len(mates)-1})")

ax.scatter(mates.loc[is_palestine, "fatality_rate_per_100k"],
           [0],
           s=260, color="#e74c3c", edgecolor="black", linewidth=1.5, zorder=5,
           label="Palestine")

# Annotate Palestine with an arrow.
pal_x = mates.loc[is_palestine, "fatality_rate_per_100k"].iloc[0]
ax.annotate("Palestine\n(4.7 / 100k — lowest in cluster)",
            xy=(pal_x, 0), xytext=(pal_x + 4, 0.6),
            fontsize=11, weight="bold", color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.5))

# Annotate the cluster average.
cluster_mean = mates["fatality_rate_per_100k"].mean()
ax.axvline(cluster_mean, color="#34495e", linestyle="--", alpha=0.5)
ax.text(cluster_mean, -0.85, f"Cluster average\n{cluster_mean:.1f}",
        ha="center", fontsize=9, color="#34495e")

ax.set_yticks([])
ax.set_ylim(-1.2, 1.2)
ax.set_xlabel("Fatality rate per 100,000 population", fontsize=11)
ax.set_title(f"Palestine vs her {len(mates)-1} cluster-mates — fatality rate distribution",
             fontsize=13, weight="bold", pad=15)
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# Print the bottom-5 and top-5 of the cluster for context.
print(f"\nLowest 5 fatality rates in cluster {palestine_cluster}:")
print(mates[["country", "fatality_rate_per_100k"]].head(5).to_string(index=False))
print(f"\nHighest 5 fatality rates in cluster {palestine_cluster}:")
print(mates[["country", "fatality_rate_per_100k"]].tail(5).to_string(index=False))

### Reading the strip plot

This is the single chart we'll put on slide 3 of the presentation. It says one thing very clearly: **Palestine has the lowest road fatality rate of any country in her structural peer group.**

The cluster average is around 19 deaths per 100k. Palestine is at 4.7 — roughly a quarter of the cluster average, and lower than her closest cluster-mate (Kiribati at 6.2). The countries at the top end of the cluster are at 30+ per 100k.

This is the kind of finding that needs the cluster context to make sense. If we just compared Palestine to *all* 172 countries globally, she'd look "good but unremarkable" — she'd sit alongside Sweden and Norway. The clustering tells us something more interesting: among countries with **similar income, similar infrastructure, and similar pedestrian-heavy mobility patterns**, Palestine is doing remarkably well on outcomes. That's worth investigating in the README and in the time-series notebook (does the rate stay this low over time?).

In [ ]:
# Heatmap — every cluster × every feature, color = how far above/below global average.
# Z-scores: positive (red) = above global mean, negative (blue) = below.

heatmap_features = [
    "fatality_rate_per_100k",
    "pct_deaths_pedestrians",
    "pct_deaths_2_3_wheelers",
    "pct_deaths_light_vehicles",
    "pct_deaths_cyclists",
    "pct_deaths_other",
    "vehicles_per_100k",
    "speed_limit_urban",
    "speed_limit_rural",
    "speed_limit_motorway",
    "bac_limit_general",
    "income_ordinal",
]

# Z-score each feature globally, then take cluster means → tells us "how typical is this cluster on this feature".
z_per_country = (features[heatmap_features] - features[heatmap_features].mean()) / features[heatmap_features].std()
z_per_cluster = z_per_country.groupby(features["cluster"]).mean()

# Friendly labels for the y-axis.
nice_names = {
    "fatality_rate_per_100k":     "Fatality rate /100k",
    "pct_deaths_pedestrians":     "% deaths pedestrians",
    "pct_deaths_2_3_wheelers":    "% deaths 2/3-wheelers",
    "pct_deaths_light_vehicles":  "% deaths light vehicles",
    "pct_deaths_cyclists":        "% deaths cyclists",
    "pct_deaths_other":           "% deaths other",
    "vehicles_per_100k":          "Vehicles per 100k",
    "speed_limit_urban":          "Speed limit urban",
    "speed_limit_rural":          "Speed limit rural",
    "speed_limit_motorway":       "Speed limit motorway",
    "bac_limit_general":          "BAC limit",
    "income_ordinal":             "Income (encoded)",
}

# Cluster labels for the x-axis (with Palestine marker).
cluster_xlabels = [f"C{i}\n{CLUSTER_NAMES[i]}" for i in sorted(z_per_cluster.index)]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    z_per_cluster.T.rename(index=nice_names),
    cmap="RdBu_r", center=0, vmin=-1.5, vmax=1.5,
    annot=True, fmt=".2f", linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Z-score vs global average\n(red = above, blue = below)"},
    ax=ax,
)
ax.set_xticklabels(cluster_xlabels, rotation=0, ha="center", fontsize=9)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Cluster signatures — how each cluster differs from the global average",
             fontsize=13, weight="bold", pad=12)

plt.tight_layout()
plt.show()

### Reading the heatmap

This chart is the master reference — every conclusion about every cluster should be visible in this one image.

Reading down the column for **Cluster 3 (Palestine)**: above-average fatality rate, strongly above-average pedestrian deaths, below-average 2-wheeler deaths, below-average vehicles per 100k, below-average income. That's the developing-pedestrian-heavy signature in numbers.

Compare with **Cluster 1 (high-income safe)**: below-average fatality rate, above-average vehicles per 100k, above-average motorway speed limit, above-average income. Mirror image.

The income row at the bottom is the cleanest dimension — it almost perfectly separates the clusters into a wealth hierarchy. This confirms what we suspected when adding income as an ordinal feature: it's a strong driver but not redundant with the death-distribution patterns above it.

One pattern worth flagging for the limitations note: **Cluster 4 looks unusually balanced** — most of its z-scores are close to zero. That's not because the cluster is "average", it's because it's the most heterogeneous group, and the within-cluster variance washes out the cluster mean. Australia ending up here (with otherwise high-income characteristics) is a symptom of this — we explore it in the limitations note below.

### Note on a data quirk: Australia

When we looked at the cluster output, we noticed Australia (high-income, very low fatality rate of 4.5) ended up in Cluster 4 alongside lower-middle-income countries — not in Cluster 1 with other wealthy countries.

We investigated. Australia has zero missing values in our dataset, so this isn't an imputation artifact. The actual cause is in the WHO file itself: Australia reports **0% of road deaths under "light vehicles"** and **67% under "other"**. Thirteen other countries (Algeria, Ethiopia, Thailand, Maldives, and others) do the same thing — they categorize their road deaths differently from the WHO questionnaire's intent, lumping most deaths into "other" rather than splitting by vehicle type.

K-Means doesn't know that "67% other" really means "67% car-driver, reported under a different label" — it just sees a data point with a weird shape and clusters it accordingly. The clustering is doing exactly what it should given the data; the data is what it is.

We document this rather than "fix" it, because making editorial decisions about what each country *should* have reported would be bigger overreach than living with the artifact. This will be flagged in the README's limitations section.

In [ ]:
# 1) country_features.csv — the full feature matrix for downstream notebooks.
features_path = PROCESSED / "country_features.csv"
features.to_csv(features_path, index=True)
print(f"Saved {features_path}")
print(f"  Shape: {features.shape}")
print()

# 2) country_clusters.csv — slim lookup table for the Streamlit map page.
clusters_out = features[[
    "country", "region", "income_group", "cluster",
    "fatality_rate_per_100k", "fatalities_count", "population",
]].copy()

# Attach human-readable cluster names so the app can label legends without re-deriving them.
clusters_out["cluster_name"] = clusters_out["cluster"].map(CLUSTER_NAMES)

clusters_path = PROCESSED / "country_clusters.csv"
clusters_out.to_csv(clusters_path, index=True)
print(f"Saved {clusters_path}")
print(f"  Shape: {clusters_out.shape}")
print()

# Verification — read it back and confirm Palestine is there with the right cluster.
check = pd.read_csv(clusters_path, index_col="iso")
print("Verification — Palestine row in country_clusters.csv:")
check.loc["PSE"]

## Handoff

Both files are now in `data/processed/` and committed to the `clustering` branch. Once this branch is merged to `main`, Osama and Omar can pull them.

**For Osama (`02_pca_anomaly.ipynb`):** Read `country_features.csv` indexed by `iso`. The 12 columns in `CLUSTER_FEATURES` (everything except population and fatalities_count) are already standardization-ready. Pass them straight into PCA and Isolation Forest.

**For Omar (`03_timeseries_forecast.ipynb`):** You don't need this file directly, but `country_clusters.csv` gives you the cluster label per country if you want to compute "Palestine vs cluster average" trend lines for the Streamlit app's Trend Analysis page.

**For the Streamlit app:** Use `country_clusters.csv` for the world map (color by `cluster_name`, label by `country`). Use `country_features.csv` for the radar chart on the Country Comparison page — the radar code in section 9.5 above can be ported almost line-for-line.

**Known limitations to mention in the README:**
- BAC limit was median-imputed for ~38 countries including Palestine. Most countries use 0.05 anyway, so the imputed value is reasonable.
- Motorway speed limit was missing for 46 countries — many likely don't have motorways. Median imputation here is a compromise.
- 14 countries (notably Australia) report road deaths in the "other" category rather than splitting by vehicle type. This causes some unexpected cluster placements, documented in section 9.5 above. We chose to leave the data as reported rather than make editorial corrections.
- The clustering reflects WHO 2023 reporting; some indicators are self-reported by countries and quality varies.
- The forecast horizon (in Omar's notebook) is statistical projection, not prediction.

---

*End of notebook 01 — Basmala, Data Mining Final Project, May 2026.*